In [1]:
import pandas as pd
import numpy as np
import torch
from tfmplayground import NanoTabPFNClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

In [2]:
# Check PyTorch and CUDA availability
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

PyTorch version: 2.9.0+cu128
CUDA available: True
CUDA device count: 8


In [3]:
# Load data
df = pd.read_csv('/home/lbote/beegfs/projects/Tabular_Foundation_Models/TFM-Playground/examples/synthetic_dataset_expanded.csv')
print(df.shape)
print(df.info())

(200, 8)
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   F1      177 non-null    float64
 1   F2      188 non-null    str    
 2   F3      184 non-null    float64
 3   F4      178 non-null    float64
 4   F5      178 non-null    str    
 5   F6      172 non-null    float64
 6   F7      179 non-null    float64
 7   Target  200 non-null    str    
dtypes: float64(5), str(3)
memory usage: 14.0 KB
None


In [4]:
df.head(10)

,F1,F2,F3,F4,F5,F6,F7,Target
0,3.75,c,0.0,46.0,NaN,2.0,1.7688,no
1,9.51,b,NaN,46.0,high,4.0,0.2532,yes
2,NaN,a,1.0,50.0,low,6.0,-0.4583,yes
3,5.99,b,1.0,29.0,high,NaN,-1.9361,no
4,1.56,b,0.0,47.0,low,6.0,-1.0764,yes
5,1.56,b,1.0,NaN,high,2.0,-1.0359,no
6,0.58,b,0.0,35.0,low,3.0,0.7331,no
7,8.66,a,0.0,34.0,high,5.0,0.4400,no
8,7.74,a,0.0,NaN,low,4.0,1.9051,yes
9,NaN,c,0.0,NaN,low,5.0,0.0356,yes


In [5]:
# Split the data into features and target
X = df.drop(columns=["Target"])
y = df["Target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)
print("train:", X_train.shape, "| test:", X_test.shape)

gpu_number = 2
device = torch.device(f"cuda:{gpu_number}" if torch.cuda.is_available() else "cpu")
categorical_features_indices = [1, 2, 4, 5]   # F2, F3, F5, F6

train: (150, 7) | test: (50, 7)


In [15]:
clf = NanoTabPFNClassifier(
    device=device, 
    num_mem_chunks=1,
    categorical_features=categorical_features_indices,
    infer_categorical=False,
)

In [17]:
clf.fit(X_train, y_train)
proba = clf.predict_proba(X_test)
print("classes_:", clf.classes_)
print("accuracy:", accuracy_score(y_test, clf.predict(X_test)))
print("roc_auc :", roc_auc_score((y_test == clf.classes_[1]).astype(int), proba[:, 1]))

classes_: ['no' 'yes']
accuracy: 0.56
roc_auc : 0.5899999999999999


In [8]:
# Auto-detección y comprobación de qué detectó
clf_auto = NanoTabPFNClassifier(device=device, 
                                num_mem_chunks=1, 
                                infer_categorical=True)
clf_auto.fit(X_train, y_train)
print("accuracy (auto):", accuracy_score(y_test, clf_auto.predict(X_test)))

def show_detected(estimator):
    for name, _, mask in estimator.feature_preprocessor.transformers_:
        cols = np.where(np.asarray(mask, dtype=bool))[0].tolist()
        print(f"  {name:3s} -> feature indices {cols}")

print("auto-detected types:")
show_detected(clf_auto) 

accuracy (auto): 0.56
auto-detected types:
  num -> feature indices [0, 3, 6]
  cat -> feature indices [1, 2, 4, 5]
  remainder -> feature indices []


In [9]:
scores = clf.feature_attention_scores(X_test)
order = np.argsort(-np.nan_to_num(scores))
for rank, idx in enumerate(order):
    print(f"{rank+1}. {X.columns[idx]:3s}  score={scores[idx]:.4f}")
# señal metida en F1, F5, F7 (y F3 flojita) -> deberían salir arriba

1. F1   score=0.1528
2. F6   score=0.1291
3. F7   score=0.1229
4. F4   score=0.1204
5. F2   score=0.1150
6. F5   score=0.1113
7. F3   score=0.1011


In [10]:
from tfmplayground.embedding import leave_one_fold_out_embeddings

emb_test = clf.get_embeddings(X_test)                                        # (50, E)
emb_train = leave_one_fold_out_embeddings(clf, X_train, y_train, n_folds=5)  # (150, E)
print("emb_test :", emb_test.shape)
print("emb_train:", emb_train.shape)

emb_test : (50, 192)
emb_train: (150, 192)


In [11]:
from sklearn.linear_model import LogisticRegression

lin = LogisticRegression(max_iter=1000).fit(emb_train, y_train)
print("lineal-sobre-embeddings:", accuracy_score(y_test, lin.predict(emb_test)))
print("nano directo           :", accuracy_score(y_test, clf.predict(X_test)))

lineal-sobre-embeddings: 0.64
nano directo           : 0.56
